# 模块 9 — Mayavi 交互式三维体渲染演示

本 notebook 演示如何在 **Jupyter 中交互式**渲染 `GoldakFDM` 的三维焊接温度场
(熔合区 / HAZ 等温面 + 对称面温度切片)。可直接用鼠标拖拽旋转、缩放、平移。

## 前置条件

安装交互式 notebook 所需依赖 (含 mayavi + jupyter):

```bash
uv sync --extra notebook
uv run jupyter lab      # 或 uv run jupyter notebook
```

> **关键**: Mayavi 的内联交互需要先调用 `mlab.init_notebook(...)`，且必须在
> **创建任何 figure 之前**执行 (见下一格)。`render(..., notebook=True)` 会返回
> figure 对象供内联显示，而不会弹出独立 GUI 窗口。

## 0. 选择内联后端

- `'x3d'` — 轻量交互后端 (基于 x3dom)，零额外依赖，可旋转/缩放；**推荐**。
- `'ipy'` — 基于 `ipyvtklink`/`ipywidgets` 的完整 VTK 交互 (更流畅，依赖更重)。
- `'png'` — 静态图，无交互 (无显示环境时的回退)。

In [ ]:
from mayavi import mlab

mlab.init_notebook('x3d')   # 改成 'ipy' 可获得更流畅交互; 'png' 为静态回退

## 1. 求解三维瞬态温度场

用默认工艺参数运行 `GoldakFDM` (半对称模型)。`run()` 同时保存末时刻温度场
`g.T` 与逐步累积的峰值温度场 `g.peak` (用于划分熔合区 / HAZ)。

In [ ]:
from welding_dynamics import GoldakFDM

g = GoldakFDM()
g.run(t_end=5.0)

L, W, D = g.pool_size()
print(f'网格 {g.Nx}x{g.Ny}x{g.Nz} (半模型)')
print(f'熔池 L×W×D = {L:.1f}×{W:.1f}×{D:.1f} mm')
print(f'峰值温度 T_max = {g.peak.max():.0f} K')

## 2. 交互渲染峰值温度场 (熔合区 + HAZ)

`render(g, field='peak', notebook=True)` 将半模型沿 y=0 镜像为全熔池，绘制
熔点等温面 (红，熔合线) 与 HAZ 等温面 (橙，半透明) 及对称面温度切片，
并返回 figure 内联显示。**用鼠标拖拽旋转视角。**

In [ ]:
from welding_dynamics import render

fig = render(g, field='peak', notebook=True)
fig

## 3. 交互渲染末时刻温度场

同一函数换成 `field='final'` 渲染热源扫过后某一时刻的温度分布。

In [ ]:
fig2 = render(g, field='final', notebook=True)
fig2

## 4. 自定义场景 (底层 mlab API)

若想交互式调节等温面阈值 / 透明度 / 增加体渲染，可直接用 `mlab` 流水线
搭建场景。下面把半模型镜像为全场后，叠加可调的多条等温面。

In [ ]:
import numpy as np

# 半模型 (y>=0) 沿 y=0 镜像为全场; z 取负 => 向下为深度
T = g.peak
Tfull = np.concatenate([T[:, :0:-1, :], T], axis=1)
x = g.x * 1e3
y = np.concatenate([-g.y[:0:-1], g.y]) * 1e3
z = -g.z * 1e3
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

fig3 = mlab.figure(bgcolor=(1, 1, 1), fgcolor=(0, 0, 0), size=(900, 600))
src = mlab.pipeline.scalar_field(X, Y, Z, Tfull)

# 多条等温面: 熔点 / 1500 / 1073 K(HAZ)
mlab.pipeline.iso_surface(src, contours=[float(g.Tm)], color=(0.85, 0.1, 0.1), opacity=0.6)
mlab.pipeline.iso_surface(src, contours=[1500.0], color=(1.0, 0.5, 0.0), opacity=0.3)
mlab.pipeline.iso_surface(src, contours=[1073.0], color=(1.0, 0.85, 0.2), opacity=0.2)

# 纵向(顶面)切片, 便于读熔深
mlab.pipeline.image_plane_widget(src, plane_orientation='z_axes',
                                 slice_index=0, colormap='jet')
mlab.colorbar(title='T [K]', orientation='vertical', nb_labels=6)
mlab.view(azimuth=-60, elevation=70, distance='auto')
fig3

## 相关

- **脚本/无头环境**: 用 `render(g, offscreen=True, outfile='results/m9.png')` 离屏存图;
  或命令行 `uv run welding-sim-3d` (求解 → 导出 OpenFOAM → 离屏截图)。
- **OpenFOAM 导出**: `from welding_dynamics import export_openfoam; export_openfoam(g)`
  写出可在 ParaView 直接打开的算例 (`results/openfoam_case/case.foam`)。